# The mathematics of SVM (hard margin)
We check the decision rule, the scaling experiment and the margin formula with small numbers.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.svm import SVC

## 1. The decision rule: positive if w · u + b ≥ 0

In [ ]:
# the line 2x + 3y + 3 = 0 written as w = (2, 3), b = 3
w = np.array([2, 3])
b = 3
for u in [np.array([3, 2]), np.array([-2, 0])]:
    value = w @ u + b                        # w · u + b
    print(u, "->", value, "positive" if value >= 0 else "negative")

## 2. Multiplying w and b by a factor: the same line, a different margin
Move the slider: pi+ and pi- (dashed) close in when k grows and spread out when k shrinks. The middle line never moves.

In [ ]:
xs = np.array([-8, 8])
ks = [0.1, 0.2, 0.5, 1, 2, 5, 10]
fig = go.Figure()
for k in ks:
    wk, bk = k * np.array([2, 3]), k * 3     # k(2x + 3y + 3)
    for rhs, colour, dash in [(0, "black", "solid"), (1, "#54A24B", "dash"), (-1, "#E45756", "dash")]:
        # y of wk[0] x + wk[1] y + bk = rhs
        fig.add_trace(go.Scatter(x=xs, y=(rhs - bk - wk[0] * xs) / wk[1], mode="lines", visible=(k == 1),
                                 line=dict(color=colour, dash=dash, width=3), showlegend=False))
steps = []
for i, k in enumerate(ks):
    visible = [j // 3 == i for j in range(3 * len(ks))]   # 3 lines per k
    margin = 2 / np.linalg.norm(k * np.array([2, 3]))
    steps.append(dict(method="update", label=str(k), args=[{"visible": visible}, {"title": f"k = {k}: margin 2/|w| = {margin:.3f}"}]))
fig.update_layout(sliders=[dict(active=ks.index(1), steps=steps, currentvalue=dict(prefix="k = "))],
                  title="k = 1: margin 2/|w| = 0.555", template="simple_white", width=650, height=600,
                  xaxis=dict(range=[-8, 8]), yaxis=dict(range=[-8, 8], scaleanchor="x"))
fig.show()

In [ ]:
for k in [10, 1, 0.1]:
    print(f"k = {k:>4}: margin 2/|w| = {2 / np.linalg.norm(k * np.array([2, 3])):.4f}")

## 3. The constraint y_i (w · x_i + b) ≥ 1 and the margin 2/|w|

In [ ]:
G = np.array([(2, 6), (3.5, 7.5), (4.5, 6), (6, 7.5), (2.5, 8.5), (5, 9), (7, 9), (7.5, 6.8)])
R = np.array([(1, 1.5), (2.5, 3), (3.5, 1), (5, 2.5), (6.5, 1.5), (7, 3.5), (1.5, 3.8), (4, 3.5)])
X = np.r_[G, R]                                   # all points
y = np.r_[np.ones(len(G)), -np.ones(len(R))]      # labels +1 (green) and -1 (red)

# a huge C makes scikit-learn's SVC behave like the hard-margin SVM
svm = SVC(kernel="linear", C=1e6).fit(X, y)
w, b = svm.coef_[0], svm.intercept_[0]
print("w =", w.round(3), " b =", round(b, 3))
print("y_i (w · x_i + b):", (y * (X @ w + b)).round(2))
print("support vectors:", svm.support_vectors_.tolist())
print("margin 2/|w| =", round(2 / np.linalg.norm(w), 3))

## 4. One outlier breaks the hard margin

In [ ]:
outlier = np.array([4.5, 2.0])                    # a green point among the red ones
print("y (w · x + b) for the outlier:", round(1 * (outlier @ w + b), 2), "(the constraint needs at least 1)")